In [104]:
import duckdb
import pandas as pd

In [105]:
con = duckdb.connect("../../data/research.duckdb")

In [106]:
PRICE_GLOB = "../../data/research-datasets/latent-fair-value/btc/price_snapshots/*.parquet"
CTX_GLOB = "../../data/research-datasets/latent-fair-value/btc/asset_context_snapshots/*.parquet"
OUT_PATH = "../../data/processed/btc_features_1m.parquet"

In [107]:
price_schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{PRICE_GLOB}')
""").df()

In [108]:
ctx_schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{CTX_GLOB}')
""").df()

In [109]:
price_schema, ctx_schema

(                           column_name column_type null   key default extra
 0                                   id      BIGINT  YES  None    None  None
 1                                asset     VARCHAR  YES  None    None  None
 2                measurement_timestamp      BIGINT  YES  None    None  None
 3                   filtered_timestamp      BIGINT  YES  None    None  None
 4                 observed_market_type     VARCHAR  YES  None    None  None
 5                       observed_asset     VARCHAR  YES  None    None  None
 6                   observed_bid_price      DOUBLE  YES  None    None  None
 7                    observed_bid_size      DOUBLE  YES  None    None  None
 8                   observed_ask_price      DOUBLE  YES  None    None  None
 9                    observed_ask_size      DOUBLE  YES  None    None  None
 10                  observed_mid_price      DOUBLE  YES  None    None  None
 11                 observed_microprice      DOUBLE  YES  None    None  None

In [110]:
price_preview = con.sql(f"""
    SELECT * FROM read_parquet('{PRICE_GLOB}') LIMIT 5
""").df()

In [111]:
ctx_preview = con.sql(f"""
    SELECT * FROM read_parquet('{CTX_GLOB}') LIMIT 5
""").df()

price_preview, ctx_preview

(   id asset  measurement_timestamp  filtered_timestamp observed_market_type  \
 0   1   BTC          1774832911478       1774832911478                 perp   
 1   2   BTC          1774832911555       1774832911555                 perp   
 2   3   BTC          1774832911609       1774832911609                 perp   
 3   4   BTC          1774832911609       1774832911609                 spot   
 4   5   BTC          1774832911611       1774832911611                 spot   
 
   observed_asset  observed_bid_price  observed_bid_size  observed_ask_price  \
 0            BTC             66399.0            6.40852             66400.0   
 1            BTC             66399.0           11.17611             66400.0   
 2            BTC             66399.0           18.71978             66400.0   
 3            BTC             66387.0            0.45573             66396.0   
 4            BTC             66387.0            0.51814             66397.0   
 
    observed_ask_size  ...  midprice

In [112]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_rows
    FROM read_parquet('{PRICE_GLOB}')
""").df()

,n_rows
0,20212345


In [113]:
con.sql(f"""
    SELECT * FROM read_parquet('{PRICE_GLOB}') LIMIT 20
""").df()

,id,asset,measurement_timestamp,filtered_timestamp,observed_market_type,observed_asset,observed_bid_price,observed_bid_size,observed_ask_price,observed_ask_size,...,midprice_basis,microprice_1p5x_filtered_timestamp,microprice_1p5x_filtered_price,microprice_1p5x_basis,microprice_3x_filtered_timestamp,microprice_3x_filtered_price,microprice_3x_basis,recorded_at_ms,source_archive_key,source_etag
0,1,BTC,1774832911478,1774832911478,perp,BTC,66399.0,6.40852,66400.0,3.72496,...,12.000000,1774832911478,66387.526477,12.105933,1774832911478,66387.526477,12.105933,1774832911780,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
1,2,BTC,1774832911555,1774832911555,perp,BTC,66399.0,11.17611,66400.0,1.18187,...,12.000000,1774832911555,66387.602357,12.297141,1774832911555,66387.601947,12.296588,1774832911789,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
2,3,BTC,1774832911609,1774832911609,perp,BTC,66399.0,18.71978,66400.0,1.18187,...,12.000000,1774832911609,66387.613806,12.325775,1774832911609,66387.613665,12.325897,1774832911842,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
3,4,BTC,1774832911609,1774832911609,spot,BTC,66387.0,0.45573,66396.0,0.12260,...,8.012068,1774832911609,66394.062888,5.876738,1774832911609,66394.033944,5.905663,1774832911905,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
4,5,BTC,1774832911611,1774832911611,spot,BTC,66387.0,0.51814,66397.0,0.02000,...,7.788179,1774832911611,66395.186751,4.755369,1774832911611,66395.166912,4.773963,1774832911912,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
5,6,BTC,1774832911611,1774832911611,perp,BTC,66399.0,18.88984,66400.0,0.00017,...,7.787687,1774832911611,66395.190618,4.792224,1774832911611,66395.170883,4.811591,1774832911950,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
6,7,BTC,1774832911689,1774832911689,perp,BTC,66399.0,21.69098,66400.0,0.00017,...,7.787525,1774832911689,66395.195390,4.804298,1774832911689,66395.175757,4.823924,1774832911970,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
7,8,BTC,1774832911689,1774832911689,spot,BTC,66387.0,0.51814,66401.0,0.07507,...,7.297414,1774832911689,66396.029128,3.972616,1774832911689,66395.978223,4.022502,1774832911997,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
8,9,BTC,1774832911869,1774832911869,spot,BTC,66395.0,0.12408,66401.0,0.67779,...,4.802543,1774832911869,66395.963951,4.018980,1774832911869,66395.948217,4.047269,1774832912087,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7
9,10,BTC,1774832911869,1774832911869,perp,BTC,66399.0,21.75250,66400.0,0.00017,...,3.493309,1774832911869,66395.965749,4.034102,1774832911869,66395.948739,4.051217,1774832912106,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,e5f28d114c1ceb6d55e1421cbe95a4c9-7


In [114]:
query = f"""
COPY (
WITH price_ranked AS (
    SELECT
        date_trunc('minute', to_timestamp(measurement_timestamp / 1000.0)) AS ts_min,
        to_timestamp(measurement_timestamp / 1000.0) AS price_ts,
        measurement_timestamp AS price_measurement_timestamp,
        spot_price,
        perp_price,
        spot_microprice,
        perp_microprice,
        basis,
        midprice_basis,
        microprice_1p5x_basis,
        microprice_3x_basis,
        row_number() OVER (
            PARTITION BY date_trunc('minute', to_timestamp(measurement_timestamp / 1000.0))
            ORDER BY measurement_timestamp DESC
        ) AS rn
    FROM read_parquet('{PRICE_GLOB}')
    WHERE lower(asset) = 'btc'
      AND spot_price IS NOT NULL
      AND perp_price IS NOT NULL
),

price_1m AS (
    SELECT
        ts_min,
        price_ts,
        price_measurement_timestamp,
        spot_price,
        perp_price,
        spot_microprice,
        perp_microprice,
        basis,
        midprice_basis,
        microprice_1p5x_basis,
        microprice_3x_basis
    FROM price_ranked
    WHERE rn = 1
),

ctx_clean AS (
    SELECT
        to_timestamp(recorded_at_ms / 1000.0) AS ctx_ts,
        recorded_at_ms,
        funding_rate,
        open_interest,
        oracle_price,
        mark_price,
        mid_price AS ctx_mid_price,
        premium,
        impact_bid_price,
        impact_ask_price,
        day_notional_volume
    FROM read_parquet('{CTX_GLOB}')
    WHERE lower(asset) = 'btc'
      AND lower(observed_market_type) = 'perp'
      AND recorded_at_ms IS NOT NULL
)

SELECT
    p.ts_min AS timestamp,
    p.price_measurement_timestamp,
    c.recorded_at_ms AS ctx_recorded_at_ms,
    p.spot_price,
    p.perp_price,
    p.spot_microprice,
    p.perp_microprice,
    p.basis AS basis_dollar,
    10000.0 * p.basis / NULLIF(p.spot_price, 0) AS basis_bps,
    p.midprice_basis,
    p.microprice_1p5x_basis,
    p.microprice_3x_basis,
    c.funding_rate,
    c.open_interest,
    c.oracle_price,
    c.mark_price,
    c.ctx_mid_price,
    c.premium,
    c.impact_bid_price,
    c.impact_ask_price,
    c.day_notional_volume
FROM price_1m p
ASOF LEFT JOIN ctx_clean c
    ON p.price_ts >= c.ctx_ts
ORDER BY timestamp
) TO '{OUT_PATH}' (FORMAT PARQUET);
"""

In [126]:
con.execute(query)
print("wrote:", OUT_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote: ../../data/processed/btc_features_1m.parquet


In [127]:
df = pd.read_parquet("../../data/processed/btc_features_1m.parquet")

In [128]:
df.tail()

,timestamp,price_measurement_timestamp,ctx_recorded_at_ms,spot_price,perp_price,spot_microprice,perp_microprice,basis_dollar,basis_bps,midprice_basis,...,microprice_3x_basis,funding_rate,open_interest,oracle_price,mark_price,ctx_mid_price,premium,impact_bid_price,impact_ask_price,day_notional_volume
2507,2026-04-01 10:56:00+00:00,1775041019928,1.775041e+12,68434.5,68400.5,68434.000736,68400.011892,-33.806026,-4.939910,-33.806026,...,-33.599460,-0.000005,27961.78370,68445.0,68410.0,68409.5,-0.000511,68400.1,68410.0,3.178871e+09
2508,2026-04-01 10:57:00+00:00,1775041079955,1.775041e+12,68502.5,68478.5,68502.565397,68478.020901,-23.998341,-3.503280,-23.998341,...,-24.548228,-0.000005,27972.94828,68515.0,68482.0,68482.5,-0.000467,68473.4,68483.0,3.181656e+09
2509,2026-04-01 10:58:00+00:00,1775041139746,1.775041e+12,68521.5,68486.5,68521.615507,68486.829618,-34.768664,-5.074125,-34.768664,...,-34.418200,-0.000005,27996.46888,68521.0,68487.0,68486.5,-0.000496,68486.0,68487.0,3.185074e+09
2510,2026-04-01 10:59:00+00:00,1775041199945,1.775041e+12,68571.5,68546.5,68571.303696,68546.013017,-25.000000,-3.645830,-25.000000,...,-25.297112,-0.000004,27982.18754,68581.0,68547.0,68546.5,-0.000496,68544.4,68547.0,3.186706e+09
2511,2026-04-01 11:00:00+00:00,1775041201469,1.775041e+12,68575.5,68546.5,68576.059246,68546.281569,-26.026229,-3.795266,-26.026229,...,-26.867892,-0.000004,27982.18754,68581.0,68547.0,68546.5,-0.000496,68542.0,68547.0,3.186706e+09


In [118]:
df.columns

Index(['timestamp', 'price_measurement_timestamp', 'ctx_recorded_at_ms',
       'spot_price', 'perp_price', 'spot_microprice', 'perp_microprice',
       'basis_dollar', 'basis_bps', 'midprice_basis', 'microprice_1p5x_basis',
       'microprice_3x_basis', 'funding_rate', 'open_interest', 'oracle_price',
       'mark_price', 'ctx_mid_price', 'premium', 'impact_bid_price',
       'impact_ask_price', 'day_notional_volume'],
      dtype='object')

In [119]:
df["open_interest"].isna().count()

2512

In [120]:
con.sql(f"""
WITH price_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{PRICE_GLOB}')
),
ctx_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{CTX_GLOB}')
)
SELECT
    'price' AS table_name,
    min_ts,
    max_ts
FROM price_check

UNION ALL

SELECT
    'context' AS table_name,
    min_ts,
    max_ts
FROM ctx_check
""").df()

,table_name,min_ts,max_ts
0,price,1774832911478,1775041201469
1,context,<NA>,<NA>


In [121]:
con.sql(f"""
WITH price_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{PRICE_GLOB}')
),
ctx_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{CTX_GLOB}')
)
SELECT
    'price' AS table_name,
    min_ts,
    max_ts
FROM price_check

UNION ALL

SELECT
    'context' AS table_name,
    min_ts,
    max_ts
FROM ctx_check
""").df()

,table_name,min_ts,max_ts
0,price,1774832911478,1775041201469
1,context,<NA>,<NA>


In [122]:
con.sql(f"""
SELECT
    measurement_timestamp,
    to_timestamp(measurement_timestamp) AS as_seconds,
    to_timestamp(measurement_timestamp / 1000.0) AS as_ms_wrong
FROM read_parquet('{CTX_GLOB}')
LIMIT 5
""").df()

,measurement_timestamp,as_seconds,as_ms_wrong
0,<NA>,NaT,NaT
1,<NA>,NaT,NaT
2,<NA>,NaT,NaT
3,<NA>,NaT,NaT
4,<NA>,NaT,NaT


In [123]:
con.sql(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(measurement_timestamp) AS nonnull_measurement_timestamp,
    COUNT(recorded_at_ms) AS nonnull_recorded_at_ms,
    COUNT(funding_rate) AS nonnull_funding_rate,
    COUNT(open_interest) AS nonnull_open_interest
FROM read_parquet('{CTX_GLOB}')
""").df()

,n_rows,nonnull_measurement_timestamp,nonnull_recorded_at_ms,nonnull_funding_rate,nonnull_open_interest
0,1721683,0,1721683,1721683,1721683


In [124]:
con.sql(f"""
SELECT
    recorded_at_ms,
    to_timestamp(recorded_at_ms / 1000.0) AS ctx_ts,
    funding_rate,
    open_interest,
    mark_price,
    oracle_price
FROM read_parquet('{CTX_GLOB}')
WHERE lower(asset) = 'btc'
  AND lower(observed_market_type) = 'perp'
LIMIT 10
""").df()

,recorded_at_ms,ctx_ts,funding_rate,open_interest,mark_price,oracle_price
0,1774890704670,2026-03-30 13:11:44.670000-04:00,-0.000016,27799.81850,67195.0,67238.0
1,1774890705373,2026-03-30 13:11:45.373000-04:00,-0.000016,27799.82444,67195.0,67238.0
2,1774890706377,2026-03-30 13:11:46.377000-04:00,-0.000016,27799.89038,67195.0,67238.0
3,1774890707379,2026-03-30 13:11:47.379000-04:00,-0.000016,27799.89038,67195.0,67238.0
4,1774890708426,2026-03-30 13:11:48.426000-04:00,-0.000016,27799.94066,67195.0,67238.0
5,1774890709510,2026-03-30 13:11:49.510000-04:00,-0.000016,27799.94706,67195.0,67238.0
6,1774890710550,2026-03-30 13:11:50.550000-04:00,-0.000016,27800.59284,67195.0,67238.0
7,1774890711410,2026-03-30 13:11:51.410000-04:00,-0.000016,27800.56496,67195.0,67238.0
8,1774890712401,2026-03-30 13:11:52.401000-04:00,-0.000016,27800.56496,67195.0,67238.0
9,1774890713414,2026-03-30 13:11:53.414000-04:00,-0.000016,27800.57124,67195.0,67238.0


In [125]:
len(df), df["timestamp"].min(), df["timestamp"].max()

(2512,
 Timestamp('2026-03-30 01:08:00+0000', tz='UTC'),
 Timestamp('2026-04-01 11:00:00+0000', tz='UTC'))